# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AmaanJilani1/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Connect (setup, not part of the graded sections)

**Setup note:** in Colab, open the key icon (Secrets) on the left sidebar, add a secret named `HF_TOKEN` with your Hugging Face READ token, and toggle notebook access on. Never paste the token directly into a cell.

In [1]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)
import duckdb

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
else:
    import os
    HF_TOKEN = os.environ.get("HF_TOKEN")

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"
FACT_MONTH = f"{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet"
CONTENT = f"{BASE}/dim_content.parquet"
CLIENTS = f"{BASE}/dim_clients.parquet"

print("Connected. Columns confirmed in fact_content_daily_performance:")
print("report_date, client_hash_id, content_hash_id, client_has_gsc, client_has_ga4,")
print("gsc_data_available, ga4_data_available, gsc_impressions, gsc_clicks, gsc_sum_position,")
print("gsc_avg_position, ga4_pageviews, ga4_sessions, ga4_users, ga4_engaged_sessions,")
print("ga4_total_engagement_sec, sessions_organic/direct/referral/social/paid/ai, ai_* breakdown,")
print("scroll_events, month")


Connected. Columns confirmed in fact_content_daily_performance:
report_date, client_hash_id, content_hash_id, client_has_gsc, client_has_ga4,
gsc_data_available, ga4_data_available, gsc_impressions, gsc_clicks, gsc_sum_position,
gsc_avg_position, ga4_pageviews, ga4_sessions, ga4_users, ga4_engaged_sessions,
ga4_total_engagement_sec, sessions_organic/direct/referral/social/paid/ai, ai_* breakdown,
scroll_events, month


## 1. Unit of analysis + time window

**One row means:** one content item, for one client, on one calendar day — the grain of `fact_content_daily_performance` (day x client x content, keyed on `report_date`, `client_hash_id`, `content_hash_id`). This matches my lane (Refresh / Content Opportunity Scoring): I need day-level performance to detect a real trend, not a single pre-computed label like the starter CSV gave me.

**Table(s) I'll use:** `fact_content_daily_performance` (daily GSC/GA4 metrics), optionally joined to `dim_content` for content-level attributes.

**Time window:** a mid-panel month, `month=2026-03`, per the assignment's guidance — the final month (June 2026, `fact_content_daily_performance_sample.parquet`) is reserved as a sealed test month and must not be used to develop label logic, since it would double as both training data and the natural "future" outcome window.

**What I'd predict (label/proxy):** whether a content item's search performance (`gsc_impressions`) is declining within the window — computed directly from daily data (second half of the month vs. first half), rather than reusing a pre-baked label like the starter CSV's `trend_direction`.

**One thing I deliberately exclude (for now):** `fact_content_query_90d.parquet` (the query-level table). It's a different grain (content x query, not content x day) and mixing grains this early risks silently changing what a "row" means. I'll bring it in later only if I need query-level detail, and I'll re-write this contract when I do.

## 2. Fields: feature / label / context / excluded

| Field | Bucket | Why |
|---|---|---|
| `gsc_impressions` | feature | direct GSC signal, knowable at report date |
| `gsc_clicks` | feature | direct GSC signal, knowable at report date |
| `gsc_avg_position` | feature | direct GSC signal, knowable at report date |
| `report_date` | context | needed to build the window/trend, not a predictive feature itself |
| `client_hash_id` | context | needed for the client-holdout split later (per notebook 02's leakage lesson) — never a feature, since the model shouldn't key off *which client* a page belongs to |
| `content_hash_id` | context | join key to `dim_content`, not itself predictive |
| `gsc_data_available` | context | used to check row usability (Section 3), not a predictive feature |
| computed trend (second-half vs first-half `gsc_impressions`) | **label/proxy** | this is what I'm trying to predict/rank by |
| `ga4_*` / `sessions_*` / `ai_*` fields | excluded (for this lane, for now) | these are GA4 on-site behavior/traffic-source signals, not search-visibility signals — mixing them in changes the question from "is this page losing search visibility" to a broader traffic question I haven't scoped for yet |
| `fact_content_query_90d.parquet` (whole table) | excluded | different grain (content x query, not content x day); mixing it in changes the unit of analysis defined in Section 1 |

## 3. Verify it with queries (grain, counts, missing values, windows) + five features + the leakage trap

In [2]:
# Query 1 - GRAIN CHECK: one row really is one (client, content, day)?
grain_check = con.sql(f"""
    SELECT COUNT(*) AS groups_with_duplicates
    FROM (
        SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS n
        FROM read_parquet('{FACT_MONTH}')
        GROUP BY client_hash_id, content_hash_id, report_date
        HAVING COUNT(*) > 1
    )
""").df()
print("Query 1 - grain check (should be 0 duplicate groups):")
print(grain_check)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 1 - grain check (should be 0 duplicate groups):
   groups_with_duplicates
0                       0


In [3]:
# Query 2 - ROW COUNT + DATE SPAN for month=2026-03
count_span = con.sql(f"""
    SELECT COUNT(*) AS row_count,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           COUNT(DISTINCT content_hash_id) AS n_content_items
    FROM read_parquet('{FACT_MONTH}')
""").df()
print("Query 2 - row count and date span for month=2026-03:")
print(count_span)


Query 2 - row count and date span for month=2026-03:
   row_count   min_date   max_date  n_clients  n_content_items
0    9841378 2026-03-01 2026-03-31         55           331437


In [4]:
# Query 3 - AVAILABILITY CHECK using IS TRUE, on the real boolean availability columns
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
    FROM read_parquet('{FACT_MONTH}')
""").df()
print("Query 3 - availability check (IS TRUE filter):")
print(availability)
print(f"\nShare of rows with GSC data available: "
      f"{availability['gsc_available_rows'][0] / availability['total_rows'][0]:.1%}")
print(f"Share of rows with GA4 data available: "
      f"{availability['ga4_available_rows'][0] / availability['total_rows'][0]:.1%}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 3 - availability check (IS TRUE filter):
   total_rows  gsc_available_rows  ga4_available_rows
0     9841378             3611061              413966

Share of rows with GSC data available: 36.7%
Share of rows with GA4 data available: 4.2%


In [5]:
# Five features, built from the March 2026 slice, filtered to rows where GSC data is actually available.
features = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS total_impressions,               -- 1. knowable at decision moment: pure GSC history, no future data
        SUM(gsc_clicks) AS total_clicks,                          -- 2. knowable at decision moment: pure GSC history, no future data
        AVG(gsc_avg_position) AS mean_position,                   -- 3. knowable at decision moment: daily rank, already observed
        SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS ctr,  -- 4. knowable at decision moment: derived only from past clicks/impressions
        SUM(CASE WHEN report_date >= DATE '2026-03-16' THEN gsc_impressions ELSE 0 END)
          - SUM(CASE WHEN report_date < DATE '2026-03-16' THEN gsc_impressions ELSE 0 END)
          AS impressions_trend_delta                              -- 5. knowable at decision moment: second-half minus first-half, both already observed by month end
    FROM read_parquet('{FACT_MONTH}')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

print(f"Feature frame shape: {features.shape}")
features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (176738, 7)


,client_hash_id,content_hash_id,total_impressions,total_clicks,mean_position,ctr,impressions_trend_delta
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,77.0,0.0,4.074107,0.000000,-37.0
1,client_62f4a7e64f5e0096,content_c03ecafd4c999f15,10849.0,22.0,8.240351,0.002028,901.0
2,client_62f4a7e64f5e0096,content_e689bc511192751a,61.0,0.0,6.015432,0.000000,-9.0
3,client_62f4a7e64f5e0096,content_7dbc094b799e05a4,705.0,1.0,5.956862,0.001418,-91.0
4,client_62f4a7e64f5e0096,content_40b10da45f4c1cb5,50.0,0.0,12.977513,0.000000,-24.0


In [6]:
# Define the proxy label from the same window: declining if second-half impressions < first-half
features["is_declining"] = (features["impressions_trend_delta"] < 0).astype(int)
print("Declining rate (honest proxy, built from features above):")
print(features["is_declining"].mean().round(3))


Declining rate (honest proxy, built from features above):
0.377


In [7]:
# THE TRAP: add one label-derived column on purpose and watch the score jump toward perfect.
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

base_feats = ["total_impressions", "total_clicks", "mean_position", "ctr"]
X_honest = features[base_feats].fillna(0)
y = features["is_declining"]

Xh_train, Xh_test, y_train, y_test = train_test_split(X_honest, y, test_size=0.2, random_state=42)
honest_tree = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42).fit(Xh_train, y_train)
honest_score = honest_tree.score(Xh_test, y_test)
print(f"HONEST accuracy (no leak): {honest_score:.3f}")

# Now sneak in impressions_trend_delta - the exact quantity the label was derived from
leaky_feats = base_feats + ["impressions_trend_delta"]
X_leaky = features[leaky_feats].fillna(0)
Xl_train, Xl_test, _, _ = train_test_split(X_leaky, y, test_size=0.2, random_state=42)
leaky_tree = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42).fit(Xl_train, y_train)
leaky_score = leaky_tree.score(Xl_test, y_test)
print(f"LEAKY accuracy (with impressions_trend_delta included): {leaky_score:.3f}  <- jumps toward 1.0, because")
print("this column IS the thing the label was computed from - the model isn't predicting, it's reading the answer key.")

# Delete the leaked column and keep the honest number
del X_leaky
print(f"\nKeeping the honest number going forward: {honest_score:.3f}")


HONEST accuracy (no leak): 0.510
LEAKY accuracy (with impressions_trend_delta included): 1.000  <- jumps toward 1.0, because
this column IS the thing the label was computed from - the model isn't predicting, it's reading the answer key.

Keeping the honest number going forward: 0.510


## 4. Data limits

- **Unbalanced panel:** per `dim_clients` (`gsc_data_start` / `ga4_data_start`-style fields, plus the per-row `client_has_gsc`/`client_has_ga4` flags), client history depth and data-source coverage differs — some clients have GSC data but not GA4, or vice versa, so this slice cannot claim to represent all clients equally.
- **`gsc_data_available` filtering removes real rows:** Query 3 shows the exact share of rows with usable GSC data — any row where it's false was dropped from the feature frame, meaning some client/content/day combinations are structurally invisible to this analysis, not just missing at random.
- **March 2026 is one month, one point in the yearly cycle:** any seasonal SEO effects specific to March won't generalize automatically to other months without re-checking.
- **The final month (June 2026 / `fact_content_daily_performance_sample.parquet`) is sealed:** I deliberately did not look at it while building label logic, since it's the natural "future" window for any past-to-future label — using it now would let information leak backward into feature design.
- **This data cannot tell me *why* a page's trend changed** — only that it changed. Any causal story (e.g. "a refresh caused recovery") would require an actual experiment, not this observational daily-performance data.
- **Query-level detail is excluded from this slice:** because `fact_content_query_90d.parquet` was deliberately left out (Section 1), this contract cannot yet say anything about *which* search queries drove the impressions/clicks I'm aggregating.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.